# Day 06 · Data Ingestion Engineering Benchmark
Corpus: 31 pages (Transformer 15p + BERT 16p)
Strategies: A (Fixed-size 500/100), B (Paragraph-aware), C (B + Metadata Filter)
Storage: PostgreSQL + pgvector (vector 1536)
Embedding: text-embedding-3-small via OpenRouter API

In [ ]:
# Setup PostgreSQL + pgvector on Kaggle
import subprocess, sys, time
t0 = time.time()
subprocess.run("sudo apt-get update -qq && sudo apt-get install -y -qq postgresql postgresql-contrib postgresql-server-dev-all build-essential git", shell=True, check=True)
subprocess.run("cd /tmp && rm -rf pgvector && git clone --branch v0.7.0 https://github.com/pgvector/pgvector.git && cd pgvector && make && sudo make install", shell=True, check=True)
subprocess.run("sudo service postgresql restart", shell=True, check=True)
subprocess.run("sudo -u postgres psql -c "CREATE USER kaggle WITH PASSWORD 'kaggle' SUPERUSER;" 2>/dev/null || true", shell=True)
subprocess.run("sudo -u postgres psql -c "CREATE DATABASE rag_db OWNER kaggle;" 2>/dev/null || true", shell=True)
subprocess.run("sudo -u postgres psql -d rag_db -c "CREATE EXTENSION IF NOT EXISTS vector;"", shell=True, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pgvector", "psycopg2-binary", "pypdf", "requests", "tabulate"], check=True)
print(f"[✓] Setup completed in {time.time() - t0:.2f}s")

In [ ]:
# Download Corpus PDFs (≥30 pages)
import urllib.request, pypdf
urllib.request.urlretrieve("https://arxiv.org/pdf/1706.03762.pdf", "attention.pdf")
urllib.request.urlretrieve("https://arxiv.org/pdf/1810.04805.pdf", "bert.pdf")
p1 = len(pypdf.PdfReader("attention.pdf").pages)
p2 = len(pypdf.PdfReader("bert.pdf").pages)
print(f"[✓] Transformer: {p1} pages, BERT: {p2} pages. Total: {p1 + p2} content pages (≥30 required)")

In [ ]:
# Ingestion & Normalization
import unicodedata, re
def normalize_text(text):
    if not text: return ""
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"(\w+)-\s*\n\s*(\w+)", r"\1\2", text)
    text = text.replace("\r", "")
    paras = [re.sub(r"\s+", " ", p).strip() for p in text.split("\n\n")]
    return "\n\n".join([p for p in paras if p])

def load_doc(pdf_path, doc_id, source):
    reader = pypdf.PdfReader(pdf_path)
    pages = []
    for i, p in enumerate(reader.pages, start=1):
        text = normalize_text(p.extract_text() or "")
        pages.append({"doc_id": doc_id, "source": source, "page": i, "text": text})
    return pages

pages_tf = load_doc("attention.pdf", "transformer-2017", "arxiv:1706.03762")
pages_bert = load_doc("bert.pdf", "bert-2018", "arxiv:1810.04805")
all_pages = pages_tf + pages_bert
print(f"[✓] Ingested {len(all_pages)} pages")

In [ ]:
# Chunking Strategies: A (Fixed 500/100) and B (Paragraph max 800)
def chunk_fixed(pages, size=500, overlap=100):
    chunks = []
    step = size - overlap
    for p in pages:
        text = p["text"]
        idx = 0
        for s in range(0, len(text), step):
            e = min(len(text), s + size)
            sub = text[s:e].strip()
            if len(sub) >= 20 or (s == 0 and len(sub) > 0):
                chunks.append({"chunk_id": f"{p['doc_id']}_p{p['page']}_f{idx}", "doc_id": p["doc_id"], "source": p["source"], "page": p["page"], "text": sub})
                idx += 1
            if e >= len(text): break
    return chunks

def chunk_paragraph(pages, max_size=800):
    chunks = []
    for p in pages:
        paras = [x.strip() for x in p["text"].split("\n\n") if x.strip()]
        curr = []
        curr_len = 0
        idx = 0
        for para in paras:
            if len(para) > max_size:
                if curr:
                    m = "\n\n".join(curr)
                    chunks.append({"chunk_id": f"{p['doc_id']}_p{p['page']}_para{idx}", "doc_id": p["doc_id"], "source": p["source"], "page": p["page"], "text": m})
                    idx += 1; curr = []; curr_len = 0
                for sub_s in range(0, len(para), max_size - 100):
                    sub_e = min(len(para), sub_s + max_size)
                    sub_t = para[sub_s:sub_e].strip()
                    if sub_t:
                        chunks.append({"chunk_id": f"{p['doc_id']}_p{p['page']}_para{idx}", "doc_id": p["doc_id"], "source": p["source"], "page": p["page"], "text": sub_t})
                        idx += 1
                    if sub_e >= len(para): break
            elif curr_len + len(para) + 2 <= max_size:
                curr.append(para); curr_len += len(para) + 2
            else:
                m = "\n\n".join(curr)
                chunks.append({"chunk_id": f"{p['doc_id']}_p{p['page']}_para{idx}", "doc_id": p["doc_id"], "source": p["source"], "page": p["page"], "text": m})
                idx += 1; curr = [para]; curr_len = len(para)
        if curr:
            m = "\n\n".join(curr)
            chunks.append({"chunk_id": f"{p['doc_id']}_p{p['page']}_para{idx}", "doc_id": p["doc_id"], "source": p["source"], "page": p["page"], "text": m})
    return chunks

chunks_a = chunk_fixed(all_pages)
chunks_b = chunk_paragraph(all_pages)
chunks_c = chunks_b
print(f"[✓] Strategy A: {len(chunks_a)} chunks, Strategy B/C: {len(chunks_b)} chunks")

In [ ]:
# Embeddings & PgVector Upsert
import os, requests, psycopg2, json, time
from pgvector.psycopg2 import register_vector
from psycopg2.extras import execute_values

API_KEY = "your_openrouter_api_key_here"

def get_embeddings(texts, batch_size=32):
    res = []
    headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}
    for i in range(0, len(texts), batch_size):
        b = texts[i:i+batch_size]
        resp = requests.post("https://openrouter.ai/api/v1/embeddings", json={"model": "text-embedding-3-small", "input": b}, headers=headers)
        d = sorted(resp.json()["data"], key=lambda x: x["index"])
        res.extend([x["embedding"] for x in d])
        time.sleep(0.3)
    return res

conn = psycopg2.connect("dbname=rag_db user=kaggle password=kaggle host=localhost port=5432")
conn.autocommit = True
register_vector(conn)
with conn.cursor() as cur:
    cur.execute("""
    DROP TABLE IF EXISTS rag_chunks;
    CREATE TABLE rag_chunks (
        id SERIAL PRIMARY KEY,
        chunk_id VARCHAR(100),
        strategy VARCHAR(50),
        document_id VARCHAR(100),
        source TEXT,
        page INT,
        text TEXT,
        embedding vector(1536)
    );
    CREATE INDEX ON rag_chunks (strategy);
    """)

print("[+] Embedding Strategy A chunks...")
vecs_a = get_embeddings([c["text"] for c in chunks_a])
data_a = [(c["chunk_id"], "A_fixed", c["doc_id"], c["source"], c["page"], c["text"], v) for c, v in zip(chunks_a, vecs_a)]
with conn.cursor() as cur:
    execute_values(cur, "INSERT INTO rag_chunks (chunk_id, strategy, document_id, source, page, text, embedding) VALUES %s", data_a, template="(%s, %s, %s, %s, %s, %s, %s::vector)")

print("[+] Embedding Strategy B & C chunks...")
vecs_b = get_embeddings([c["text"] for c in chunks_b])
data_b = [(c["chunk_id"], "B_paragraph", c["doc_id"], c["source"], c["page"], c["text"], v) for c, v in zip(chunks_b, vecs_b)]
data_c = [(c["chunk_id"], "C_meta_filter", c["doc_id"], c["source"], c["page"], c["text"], v) for c, v in zip(chunks_c, vecs_b)]
with conn.cursor() as cur:
    execute_values(cur, "INSERT INTO rag_chunks (chunk_id, strategy, document_id, source, page, text, embedding) VALUES %s", data_b, template="(%s, %s, %s, %s, %s, %s, %s::vector)")
    execute_values(cur, "INSERT INTO rag_chunks (chunk_id, strategy, document_id, source, page, text, embedding) VALUES %s", data_c, template="(%s, %s, %s, %s, %s, %s, %s::vector)")
print("[✓] All strategies embedded and stored in PostgreSQL pgvector!")

In [ ]:
# Benchmark Evaluation on 20 Questions (15 Dev / 5 Test)
with open("benchmark_questions.json") as f:
    questions = json.load(f)

def search_db(strategy, q_vec, doc_filter=None):
    with conn.cursor() as cur:
        if doc_filter:
            cur.execute("""
            SELECT chunk_id, document_id, page, text, 1 - (embedding <=> %s::vector) AS sim
            FROM rag_chunks
            WHERE strategy = %s AND document_id = %s
            ORDER BY embedding <=> %s::vector LIMIT 3;
            """, (q_vec, strategy, doc_filter, q_vec))
        else:
            cur.execute("""
            SELECT chunk_id, document_id, page, text, 1 - (embedding <=> %s::vector) AS sim
            FROM rag_chunks
            WHERE strategy = %s
            ORDER BY embedding <=> %s::vector LIMIT 3;
            """, (q_vec, strategy, q_vec))
        return cur.fetchall()

results_summary = {}
for strat_name, strat_tag in [("Strategy A (Fixed)", "A_fixed"), ("Strategy B (Paragraph)", "B_paragraph"), ("Strategy C (Filter)", "C_meta_filter")]:
    h1_dev = 0; h3_dev = 0; h1_test = 0; h3_test = 0
    dev_count = 0; test_count = 0
    for q in questions:
        q_vec = get_embeddings([q["question"]])[0]
        doc_f = q["filter"]["document_id"] if (strat_tag == "C_meta_filter" and q["filter"]) else None
        hits = search_db(strat_tag, q_vec, doc_f)
        is_h1 = (hits[0][1] == q["gold_source_id"] and hits[0][2] == q["gold_page"])
        is_h3 = any(h[1] == q["gold_source_id"] and h[2] == q["gold_page"] for h in hits[:3])
        if q["split"] == "dev":
            dev_count += 1
            if is_h1: h1_dev += 1
            if is_h3: h3_dev += 1
        else:
            test_count += 1
            if is_h1: h1_test += 1
            if is_h3: h3_test += 1
    results_summary[strat_name] = {
        "Dev Hit@1": f"{h1_dev}/{dev_count} ({h1_dev/dev_count*100:.1f}%)",
        "Dev Hit@3": f"{h3_dev}/{dev_count} ({h3_dev/dev_count*100:.1f}%)",
        "Test Hit@1": f"{h1_test}/{test_count} ({h1_test/test_count*100:.1f}%)",
        "Test Hit@3": f"{h3_test}/{test_count} ({h3_test/test_count*100:.1f}%)",
        "Overall Hit@3": f"{(h3_dev+h3_test)/20*100:.1f}%"
    }

from tabulate import tabulate
rows = [[k, v["Dev Hit@1"], v["Dev Hit@3"], v["Test Hit@1"], v["Test Hit@3"], v["Overall Hit@3"]] for k, v in results_summary.items()]
print("\n" + "="*80)
print("BENCHMARK RESULTS ON 20 QUESTIONS (15 DEV / 5 TEST)")
print("="*80)
print(tabulate(rows, headers=["Strategy", "Dev Hit@1", "Dev Hit@3", "Test Hit@1", "Test Hit@3", "Overall Hit@3"], tablefmt="grid"))
with open("benchmark_summary.json", "w") as f: json.dump(results_summary, f, indent=2)